# Mini Project 12 — E-Commerce Refund Requests (Bronze → Silver → Gold)
### Theme: string cleaning & repair (`lpad`, `split`) + core cleaning routine

**Data:** `refund_requests_12.csv` — one table, 108 raw rows (synthetic practice dataset).
Columns: `refund_id, order_id, sku, category_path, refund_amount, request_date, status, reason`.

Take-home format: no step-by-step guide — the brief is the scenario plus three business questions,
and the steps, checks and decisions below are my own.

## Scenario

You are a data engineer at **VeloShop**, a Dutch online store. The Finance team wants a **reliable refund report**,
but the raw export from the returns tool is messy: refund amounts come in mixed currency formats, dates in several
formats, statuses are typed inconsistently, and a few rows were exported twice.

One extra detail from the catalog team: **a product SKU is always exactly 6 digits** in the catalog system
(leading zeros included) — but this export went through Excel at some point, and Excel did what Excel does to numbers.

The `category_path` column stores the full category tree as one pipe-separated string
(e.g. `Electronics|Audio|Headphones`). Finance only reports at the **main category** level.

## Task (this is the whole brief — you design the steps)

Read the raw file and turn it into a **trustworthy, analysis-ready refunds table (Silver)** — SKUs restored to their
real catalog form, a usable `main_category` column, clean amounts/dates/statuses — then answer the three business
questions below (Gold). Decide yourself what "trustworthy" means and keep a **short decision log** (a markdown cell:
what you changed, how many rows/values it touched, why). Your log and your code must tell the same story.
Don't over-engineer: clean only what the questions actually depend on.

## Step 1 — Read raw (everything as string)

Reading with no schema on purpose: on a dirty export I want to *see* the dirt first, not let Spark
guess types and silently turn broken values into nulls before I've measured them.

In [ ]:
refund_data_raw_df = (
    spark.read
    .option("Header", True)
    .format("csv")
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/refund_requests_12.csv")
)

refund_data_raw_df.display()

In [ ]:
refund_data_raw_df.printSchema()

## Step 2 — Profile before touching anything

Row counts, distinct counts and duplicate hunting **before** any cleaning ("measure first, delete later"):

- total rows vs `dropDuplicates().count()` tells me if exact copies exist
- distinct `refund_id` vs total tells me if the same id appears twice
- grouping by **all** columns proves whether duplicated ids are *exact row copies* or *conflicting records* — two very different problems

In [ ]:
from pyspark.sql.functions import col

print(refund_data_raw_df.filter(col("refund_id").isNull()).count())   # null ids -> 0
print(refund_data_raw_df.count())                                     # 108
print(refund_data_raw_df.select("refund_id").distinct().count())      # 105 -> 3 ids repeat
print(refund_data_raw_df.dropDuplicates().count())                    # 105 -> same drop => likely exact copies

# which ids repeat?
refund_data_raw_df.groupBy("refund_id").count().filter(col("count") > 1).display()

# are the repeated rows identical in EVERY column? group by all columns to prove it
refund_data_raw_df.groupBy(refund_data_raw_df.columns).count().filter(col("count") > 1).display()

# result: RF-3011, RF-3031, RF-3077 -> each appears twice, identical across all columns = exact export duplicates

## Step 3 — Deduplicate (full-row)

The 3 duplicated ids are exact full-row copies, so a full-row `dropDuplicates()` is safe: it keeps one copy
of each record and only removes the double-export. 108 → 105 rows. (If the copies had *conflicting* values,
this would have been a decision, not a cleanup.)

In [ ]:
refund_data_raw_dedup_df = refund_data_raw_df.dropDuplicates()

print(refund_data_raw_dedup_df.count())   # 105

## Step 4 — Normalize strings

Trim everything; lowercase the free-text-ish columns (`category_path`, `status`, `reason`) so case variants
(`Approved` / `APPROVED` / `approved`) collapse into one value. IDs and SKU are only trimmed — changing their
case could change the identifier itself.

In [ ]:
from pyspark.sql.functions import trim, col, lower

refund_data_normalized_df = refund_data_raw_dedup_df.withColumns({
    "refund_id": trim(col("refund_id")),
    "order_id" : trim(col("order_id")),
    "sku"      : trim(col("sku")),
    "category_path" : trim(lower(col("category_path"))),
    "refund_amount" : trim(col("refund_amount")),
    "request_date"  : trim(col("request_date")),
    "status"       : trim(lower(col("status"))),
    "reason"       : trim(lower(col("reason")))     
    
})
                                     
refund_data_normalized_df.display()

## Step 5 — Placeholders → NULL

`n/a`, `error`, `unknown` are not data — they are the export tool's way of saying "no value". Converting them
to real NULLs *before* casting means the cast can't silently swallow them and I can count exactly what is missing:
**5 refund_amount values and 9 reason values.**

(A bug I caught here the hard way: my first version of the `reason` line checked `col("status")` in the `when()`
condition — a copy-paste error. It ran without any error and simply never matched. The fix showed up only because
I re-checked the *output*, not the code.)

In [ ]:
from pyspark.sql.functions import when

refund_data_normalized_df = refund_data_normalized_df.withColumns({
    "refund_id"    :  when(lower(col("refund_id")).isin("unknown","n/a","error"),None).otherwise(col("refund_id")),
    "order_id"     :  when(lower(col("order_id")).isin("unknown","error","n/a"), None).otherwise(col("order_id")),
    "sku"          :  when(lower(col("sku")).isin("unknown","error","n/a"), None).otherwise(col("sku")),
    "category_path":  when(lower(col("category_path")).isin("unknown","error","n/a"), None).otherwise(col("category_path")),
    "refund_amount":  when(lower(col("refund_amount")).isin("unknown","error","n/a"), None).otherwise(col("refund_amount")),
    "request_date" : when(lower(col("request_date")).isin("unknown","error","n/a"), None).otherwise(col("request_date")),
    "status"       : when(lower(col("status")).isin("unknown","error","n/a"), None).otherwise(col("status")),
    "reason"       : when(lower(col("reason")).isin("unknown","error","n/a"), None).otherwise(col("reason"))
})

refund_data_normalized_df.display()

## Step 6 — Repair & type the columns

- **sku** — catalog rule: a SKU is always exactly 6 digits. Excel stripped the leading zeros, so
  `lpad(sku, 6, "0")` restores the real catalog form. This is the catalog team's rule, not my assumption.
- **category_path** — Finance reports at main-category level only: `split` on the (escaped) pipe + `getItem(0)`.
  A per-row operation: row count does not change.
- **refund_amount** — strip `€` / `EUR` / spaces, comma → dot. Clean first, cast later.
- **request_date** — four different date formats in one column → `coalesce(try_to_date × 4)`.
  Each pattern must match the raw text *exactly* (the space in `'MMM d, yyyy'` cost me a round of nulls).

In [ ]:
from pyspark.sql.functions import when, length,col,replace,lit,lpad,split,coalesce,try_to_date

refund_data_cleaned_df = refund_data_normalized_df.withColumns({
    "sku"              :  when(length(col("sku"))!=6, lpad(col("sku"),6,"0")).otherwise(col("sku")),
    "category_path"    :  split(col("category_path"),"\\|").getItem(0),
    "refund_amount"    :  replace(replace(replace(replace(col("refund_amount"),lit('€ '),lit('')),lit('€'),lit('')),lit('EUR'),lit('')),lit(','),lit('.')),
    "request_date"     :  coalesce(
        try_to_date('request_date','yyyy-M-d'),
        try_to_date('request_date','MMM d, yyyy'),
        try_to_date('request_date', 'd/M/yyyy'),
        try_to_date('request_date','yyyy/M/d')
    ),
    
})

refund_data_cleaned_df.display()

**Checks for this step** — every repair gets its proof:

In [ ]:
from pyspark.sql.functions import length

# SKU: nothing left that is not 6 digits
print(refund_data_cleaned_df.filter(length(col("sku")) != 6).count())      # 0

# dates: nothing silently dropped by the parsers
print(refund_data_cleaned_df.filter(col("request_date").isNull()).count()) # 0

The column now holds only the main category, so the name should say so — a column named
`category_path` showing `electronics` breaks the reader's trust in the schema.

In [ ]:
refund_data_cleaned_df = refund_data_cleaned_df.withColumnRenamed("category_path","category")
refund_data_cleaned_df.display()

## Step 7 — Cast amount to decimal, with a before/after proof

Null count before the cast (5) must equal null count after (5) — if the second number were bigger,
the cast would be silently killing values I failed to clean.

In [ ]:
refund_data_cleaned_df.filter(col("refund_amount").isNull()).count()   # 5 (placeholders nulled earlier)

In [ ]:
refund_data_cleaned_df = refund_data_cleaned_df.withColumn(
    "refund_amount", col("refund_amount").cast("decimal(10,2)")
)

print(refund_data_cleaned_df.filter(col("refund_amount").isNull()).count())  # still 5 -> cast lost nothing

## Step 8 — Negative amounts: measure first, then decide

Looking at *all* suspicious amounts before deciding anything:

In [ ]:
refund_data_cleaned_df.filter((col("refund_amount")<0) | (col("refund_amount").isNull())).display()

**What the measurement showed:** 5 rows with NULL amounts, and **2 rows with negative amounts —
RF-3028 (−171.86) and RF-3032 (−71.40), both `approved`, both `electronics`, both "too late".**

**Decision:** negative refund amounts are treated as invalid records and removed from Silver. The NULL-amount rows
are **kept** — their reason / status / date are still valid information, and question 3 counts *requests*, not euros.

**Trade-off I accept:** the 2 removed rows were real requests with a valid reason, so the Q3 reason counts
understate "too late" by 2. Logged, not hidden.

**Data-quality signal for the source team:** two *approved* refunds with *negative* amounts, both in one category,
looks like an upstream systematic issue (correction entries or an export bug) — worth investigating, not just deleting.

In [ ]:
refund_data_cleaned_df = refund_data_cleaned_df.filter(
    (col("refund_amount") > 0) | (col("refund_amount").isNull())
)

print(refund_data_cleaned_df.count())   # 105 - 2 negatives = 103

## Step 9 — Write Silver (idempotent)

`overwrite` replaces the table instead of appending, so re-running the notebook gives the same 103 rows
every time — no duplicated data on a rerun.

In [ ]:
refund_data_cleaned_df.write.mode("overwrite").saveAsTable("dev.spark_db.silver_refund_table_P12")

## Decision log (Silver)

| # | What | How many | Why |
|---|------|----------|-----|
| 1 | Exact full-row duplicates removed (`RF-3011`, `RF-3031`, `RF-3077`) | 108 → 105 rows | Verified identical across **all** columns first (groupBy all columns) — double export, safe to drop |
| 2 | Placeholders `n/a` / `error` / `unknown` → NULL | 5 amounts, 9 reasons | Placeholders are "no value", not data; nulling *before* the cast keeps the loss measurable |
| 3 | SKU restored with `trim` + `lpad(sku, 6, "0")` | all short SKUs; check: 0 rows ≠ 6 digits | Catalog team's rule ("always 6 digits") — Excel stripped leading zeros |
| 4 | `category_path` → first pipe segment, renamed `category` | row count unchanged | Finance reports at main-category level; full path not needed by any question |
| 5 | Amounts: `€`/`EUR`/comma dirt stripped, cast `decimal(10,2)` | nulls 5 before = 5 after cast | Clean first, cast later; before/after null counts prove the cast lost nothing |
| 6 | Dates: 4 formats parsed via `coalesce(try_to_date × 4)` | 0 nulls after parse | Every raw format matched exactly, nothing silently dropped |
| 7 | **Negative amounts removed** (`RF-3028` −171.86, `RF-3032` −71.40) | 105 → **103** rows | Invalid as cost records; both approved + electronics → flagged upstream as a DQ signal. NULL amounts **kept** (their reason/status/date are valid) |
| 8 | Silver written with `overwrite` | — | Idempotent rerun: same file in → same 103 rows out |

---
## Gold — Business questions

Answer each question on your cleaned Silver table. Write the query in the empty cell under it.

In [ ]:
silver_refund_df = spark.table("dev.spark_db.silver_refund_table_p12")

print(silver_refund_df.count())   # 103 -> matches what Silver should contain

### Q1. Approved refund cost per main category

Finance wants to know how much **approved** refunds cost per **main category**, highest first.
Before you present it: are there any amount values in this data you should *not* blindly include in a cost report?
Whatever you decide, state it in your decision log — this decision changes the ranking.

In [ ]:
# sanity check: Silver must contain no negative amounts (removed at cleaning, with rationale in the log)
silver_refund_df.filter(col("refund_amount") < 0).display()   # no rows

In [ ]:
from pyspark.sql.functions import sum, col, round

approved_refunds_per_category_df = (
    silver_refund_df
    .filter(col("status") == "approved")                       # row-level filter BEFORE groupBy
    .groupBy("category")
    .agg(round(sum(col("refund_amount")), 2).alias("total_refund"))
    .orderBy(col("total_refund").desc())
)

approved_refunds_per_category_df.display()

# No extra amount-filter needed here: negatives were already excluded in Silver (decision log #7).
# That decision matters for this exact ranking: both negatives were approved + electronics,
# so including them would cut electronics' total by ~243 and could flip the category order.

### Q2. Worst month

Which **month** had the highest total refund amount (all statuses)? Report the month in a form where
January 2025 and January 2026 could never be mixed together, and show all months sorted so the trend is visible.

In [ ]:
from pyspark.sql.functions import date_format

# Month key = yyyy-MM: January 2025 and January 2026 can never merge, and the
# alphabetical sort of the key IS the chronological sort. Added as a new column -
# overwriting request_date would destroy the day-level information.
monthly_total_refund = (
    silver_refund_df
    .withColumn("month", date_format(col("request_date"), "yyyy-MM"))
    .groupBy("month")
    .agg(sum(col("refund_amount")).alias("total_refund"))
    .orderBy(col("month"))
)

monthly_total_refund.display()

# Highest month: 2025-05 with 2342.26 total refund amount.

### Q3. Top refund reasons

What are the **top 2 reasons** customers request refunds, by number of requests?
Make sure placeholder junk doesn't show up as a "reason" in your result.

In [ ]:
(silver_refund_df
    .filter(col("reason").isNotNull())        # placeholders were nulled in Silver; a report must not list null as a "reason"
    .groupBy("reason")
    .count()
    .orderBy(col("count").desc())
    .display())

# Top 2 reasons: "too late" and "wrong item".
# Note from the log: 9 requests had no usable reason (placeholders) and are excluded from this ranking;
# the 2 removed negative-amount rows were also "too late" requests, so this count understates that bucket by 2.

---
## Interview questions — answer in English

Answer in a code cell as comments (or a markdown cell you add). Keep answers short and concrete.

1. How did you restore the broken SKUs, and how did you know what the correct form was? What is the risk of this
repair if the catalog team's "always 6 digits" claim turned out to be wrong?
2. You derived `main_category` from `category_path`. Did that change the row count of the table — and in what kind of
situation would deriving data from a column like this *require* `explode` instead?
3. Walk me through every kind of bad `refund_amount` value you found and what you did with each one. Why did the Q1
ranking depend on one of those decisions?
4. If this notebook runs again tomorrow on the same file, does your Silver table stay correct (no duplicated data,
same row count)? What exactly makes it safe — or what would you have to change to make it safe?

1. The leading zeros were stripped by Excel, so I used `lpad` to pad every short SKU back to 6 digits.
I knew the correct form because the catalog team already told me SKUs are always exactly 6 digits — it is their rule,
not my assumption. If that rule turned out to be wrong, the risk is that I would be creating SKU codes that never
existed: if `1234` was a real 4-digit SKU, my `001234` either matches nothing in the catalog, or worse, silently
matches a *different* product. Because I still keep the bronze layer, I can rerun the repair from raw data if the
rule changes — the repair is reversible.

2. No — the row count stayed the same (105 at that point), because `split` + `getItem` is a per-row operation:
one input row, one output row. The grain did not change. I would need `explode` if the question were about the
individual elements — for example "how often does each category *level* appear?" — because then I would need one
row per element, which changes what a row represents.

3. I found three kinds of bad `refund_amount` values. (1) Format dirt — `€` / `EUR` symbols and comma decimals:
stripped the symbols, converted comma to dot, then cast to `decimal(10,2)`, and compared null counts before and
after the cast (5 = 5) to prove the cast didn't silently lose anything. (2) Placeholders — `n/a` and `error`
(5 values): converted to NULL before the cast. (3) Negative amounts — 2 rows, both approved and both electronics:
removed from Silver and logged as a data-quality signal for the source team. The Q1 ranking depended on the
*negative* decision: both negatives sat in electronics, so including them would cut electronics' total by ~243
and could change the category order. Nulls were not the risk in Q1 — `sum()` ignores nulls automatically.

4. `overwrite` replaces the table instead of appending, so running the notebook twice gives the same 103 rows —
the pipeline is idempotent. One caveat I learned on this project: that only holds if the notebook itself is safe
to rerun top-to-bottom. At one point a destructive filter cell in the middle silently changed every downstream
number until I isolated the decision, measured what it dropped, and moved it to a single documented step.

---
## Key Takeaways

- **Measure first, delete later.** Every removal in this project (duplicates, placeholders, negatives) was counted
  and inspected *before* the delete — and the one time I filtered first and checked after, the check was circular
  and proved nothing.
- **A repair is only as good as its source of truth.** The SKU fix is safe because it implements the catalog team's
  rule, and reversible because bronze still exists.
- **A decision made for one report must not silently leak into others.** Excluding bad amounts globally also changed
  the *reason counts* in Q3 — every exclusion needs an owner: either the Silver contract (logged) or the single
  query that needs it.
- **Column names are part of the contract.** `category_path` holding only a main category, or a date column holding
  `May-2025` strings, breaks the next reader — rename or add a new column instead of overwriting.
- **Silent no-op bugs don't announce themselves.** The `when(col("status"))`/`reason` copy-paste error ran green and
  did nothing; only checking the output caught it.